# ⚡ 11 · 专家分页（Expert Paging）与显存池化

> AI-Infra 面试高频：**MoE 的权重明明放不进显存，为什么还能跑？** 每次缺页要多花多少时间？

- Mixtral 8x7B 的全部专家权重 ≈ **45.1 G 参数 / 90.2 GB @BF16**，单卡 80 GB 装不下；但每个 token 每层只走 **top-2 of 8** 个专家；
- 于是把 `(层, 专家)` 当作**逻辑页**，把专家权重按页映射到显存里的固定**槽位（slot）**，缺页时才从 CPU/DRAM 经 PCIe 搬进来——这就是 **专家分页 + 显存池化**；
- 它与 PagedAttention **同一个页表抽象、两套生命周期**：KV 页按请求分配、随序列增长；专家页按 `(层, 下标)` 分配、跨请求共享、随热度反复换入换出。


## 📋 本节要解决的问题

| # | 问题 | 直观答案 | 量化口径 |
|---|---|---|---|
| 1 | MoE 权重为什么不能全驻留？ | 专家太多，模型权重比 KV 缓存大一个量级 | 45.1 G 参数 × 2 B = **90.2 GB**，单卡 80 GB |
| 2 | 必须全驻留吗？ | 每个 token 每层只激活少数专家，权重是稀疏使用的 | 激活率 **top-2 / 8 = 25%**（每层） |
| 3 | 不驻留就得搬，搬一次多贵？ | 一个专家要过一遍 PCIe | 352.3 MB ÷ 25 GB/s ≈ **14.1 ms** |
| 4 | 14 ms 能忍吗？ | decode 一步本来只有十几毫秒，忍不了 | 每 token 64 个专家 → **≈902 ms/token** |
| 5 | 那怎么才能不每次都搬？ | 把热专家缓存起来，命中就零搬运 | 停顿 = miss 数 × 14.1 ms |
| 6 | 缓存满了淘汰谁？ | 最久没用（LRU）/ 用得最少（LFU）/ 先进先出 | Zipf 路由下 **LRU ≥ FIFO** |
| 7 | 和 PagedAttention 一样吗？ | 同一个页表思想，页的**生命周期**完全不同 | KV 页按请求 / 专家页按下标 + 跨请求共享 |


## 1️⃣ 直觉与大白话

**专家 = 图书馆里的书，显存 = 你手上的书架。**

- 整座图书馆有 256 本书（32 层 × 8 专家 = 256 个专家权重块），书架只能放 8 本；
- 读者（token）每次只借两三本；书不在书架上，就得去**书库（CPU/DRAM）**跑一趟，一趟约 14 ms；
- 「专家分页」就是**图书馆的调书制度**：书架编号固定（slot 0…7），目录卡（页表）记录「哪本书在哪个格子」；书还回来时把目录卡改成「不在架上」，下次再借就是一次缺页（page fault）。

⚠️ 三个容易混淆的点：

1. **槽位是地址，不是身份**：同一个 slot 这秒放 `(L0, E3)`，下一秒放 `(L5, E1)`——所以淘汰时必须**先把旧页表项置 -1**，否则会出现两个页指向同一槽位的「双租」bug（本章「⚠️ 五个最容易栽的坑」有最小复现）；
2. **专家页是只读权重**，淘汰时不需要写回磁盘（脏页写回可省）；KV 页则可能被写、且绑定请求；
3. **分页 ≠ 换层调度**：分页解决「放哪、放谁」；真正的流水线还要解决「什么时候提前搬」（预取），二者叠加才是工程方案。


### 1.1 三种驻留策略，为什么最终都走向「分页 + 热缓存」

| 策略 | 做法 | 显存需求 | 每 token 停顿 | 适用 |
|---|---|---|---|---|
| **全驻留** | 所有专家常驻显存 | 90.2 GB（放不下） | 0 | 单机多卡、大显存卡 |
| **整层换入** | 每层把 8 个专家整层搬入 | 单层 2.8 GB | 每层一次 112 MB，几乎每步都搬 | 训练 / prefill 大 batch |
| **分页 + 热缓存** | 只驻留当前热的少数页，缺页再搬 | slots × 352 MB | 命中 0，缺失 14.1 ms | **decode、在线服务** |

关键洞察：**「热」是随时间变化的**——一段代码类 prompt 会让数学专家冷下来；所以策略必须是「按页 + 按热度」，而不是「按层」。

三种策略的内存-停顿关系，画成图就是 **3️⃣ 节** 那张机制图：页表负责「谁在哪个槽位」，槽位池负责「容量」，PCIe 车道负责「缺页的代价」。


## 2️⃣ 量化账本

先把口径钉死（后面的所有数字都由这几个常数算出来，不写死漂亮结果）：

- 单个专家 = `gate`、`up`、`down` 三个矩阵，每个 $d\times d_{ff}$ → 参数量 $3\,d\,d_{ff}$，$d=4096$、$d_{ff}=14336$；
- BF16 每参数 2 字节；**MB = $10^6$ B、GB = $10^9$ B**（与 PCIe 厂商口径一致，不用 1024 进制）；
- PCIe 4.0 ×16 有效带宽按 **25 GB/s**（理论 31.5 GB/s，实测 22–26 GB/s）；
- 搬运时间（忽略协议开销与并发重叠）：

$$ t_{\text{expert}} = \frac{352.3\ \text{MB}}{25\ \text{GB/s}} \approx 14.1\ \text{ms} $$


In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
import unicodedata

import numpy as np
import matplotlib.pyplot as plt


def dwidth(s):
    '''估算显示宽度：CJK 全角字符记 2，其余记 1（中英混排表格对齐用）'''
    return sum(2 if unicodedata.east_asian_width(ch) in 'WF' else 1 for ch in str(s))


def pad(s, width, align='<'):
    '''按显示宽度补空格，避免中文表格错位'''
    s = str(s)
    gap = max(0, width - dwidth(s))
    return (' ' * gap + s) if align == '>' else (s + ' ' * gap)

# 口径：MB = 1e6 B，GB = 1e9 B（不用 1024 进制，便于和 PCIe 标称带宽对齐）
MB, GB = 1.0e6, 1.0e9

D_MODEL, D_FF = 4096, 14336      # Mixtral 8x7B：hidden size / expert intermediate size
N_LAYERS, N_EXPERTS = 32, 8      # 32 层 × 每层 8 专家 = 256 个专家权重页
TOP_K = 2                        # 每 token 每层激活 2 个专家
BYTES_BF16 = 2

# 数值实验用「缩比模型」：4 层 × 8 专家 = 32 页。
# 原因：slots ∈ {4..64} 直接套在 32 层上，等于每层 0.1~2 个专家，几乎全落在抖动区；
# 缩比后每层 1~8 个专家，命中率曲线才看得清。单专家字节数、PCIe 带宽、路由形状都不变，
# 真实 32 层时把 slots 与命中率曲线按同等思路放大即可（每层 2 个专家 ≈ 64 槽位）。
SIM_LAYERS = 4
SIM_PAGES = SIM_LAYERS * N_EXPERTS

PCIE_BW = 25.0    # GB/s：PCIe 4.0 x16 有效带宽口径
HBM_BW = 2000.0   # GB/s：A100/H100 量级 HBM 带宽（只作对照）
NVME_BW = 7.0     # GB/s：PCIe 4.0 NVMe 顺序读量级

HW_ROWS = [
    ('HBM（GPU 显存）', HBM_BW, '几十 GB，算子直接读'),
    ('NVLink / C2C（卡间）', 300.0, '跨卡，拓扑相关'),
    ('PCIe 4.0 x16（CPU↔GPU）', PCIE_BW, '专家 offload 的必经通道'),
    ('NVMe SSD', NVME_BW, '冷专家 / 权重库'),
]
print('硬件带宽口径（GB/s，本节所有停顿估算都用这张表）：')
for name, bw, note in HW_ROWS:
    print(f'  {name:<26}{bw:>8.1f} GB/s   {note}')


In [ ]:
params_per_expert = 3 * D_MODEL * D_FF                      # gate + up + down
bytes_per_expert = params_per_expert * BYTES_BF16          # BF16 下的字节数
params_all_experts = params_per_expert * N_LAYERS * N_EXPERTS
mem_all_experts = params_all_experts * BYTES_BF16

t_one_ms = bytes_per_expert / GB / PCIE_BW * 1e3           # 单专家搬运时间（ms）
t_all_s = mem_all_experts / GB / PCIE_BW                   # 全量冷加载（s）

print('Mixtral 8x7B 的专家权重账本（全部由上面的常数现算）：')
print('-' * 74)
rows = [
    ('单专家参数量', '3 · d · d_ff', f'{params_per_expert / 1e6:8.2f} M'),
    ('单专家 BF16 体积', '参数量 × 2 B', f'{bytes_per_expert / MB:8.2f} MB'),
    ('单层 8 专家参数量', '8 × 单专家', f'{params_per_expert * N_EXPERTS / 1e9:8.3f} G'),
    ('单层 8 专家体积', '8 × 单专家', f'{bytes_per_expert * N_EXPERTS / MB:8.1f} MB'),
    ('全部专家参数量', '32 层 × 8 专家', f'{params_all_experts / 1e9:8.2f} G'),
    ('全部专家 BF16 体积', '参数量 × 2 B', f'{mem_all_experts / GB:8.2f} GB'),
    ('单专家 PCIe 搬运', '体积 ÷ 25 GB/s', f'{t_one_ms:8.2f} ms'),
    ('全量冷加载', '总量 ÷ 25 GB/s', f'{t_all_s:8.2f} s'),
]
for name, formula, value in rows:
    print(pad(name, 26) + pad(formula, 26) + pad(value, 12, '>'))
print('-' * 74)
print(f'对照：90.2 GB ≈ 单卡 80 GB HBM 的 {mem_all_experts / GB / 80:.2f} 倍 → 单卡必须 offload')
print('对照：Mixtral 8x7B 总参数 ≈46.7 G，专家占 45.1 G，注意力/嵌入只有 ≈1.6 G')

# ---- 断言：区间/上下界，不做浮点等值比较 ----
assert 1.70e8 <= params_per_expert <= 1.82e8, params_per_expert
assert 340 * MB <= bytes_per_expert <= 365 * MB, bytes_per_expert
assert 44.0 <= params_all_experts / 1e9 <= 46.0, params_all_experts / 1e9
assert 88.0 <= mem_all_experts / GB <= 92.0, mem_all_experts / GB
assert 13.0 <= t_one_ms <= 15.0, t_one_ms
assert 3.4 <= t_all_s <= 3.9, t_all_s
print('\n✓ 账本断言通过：单专家≈1.76e8 参数 / ≈352 MB，总量∈[44,46] G 参数')


In [ ]:
def transfer_ms(nbytes, bw_gbps):
    '''按 GB/s 带宽估算搬运耗时（ms），忽略启动开销'''
    return nbytes / GB / bw_gbps * 1e3

print('搬运【一个专家】(352.3 MB) 在不同通道上的耗时：')
for name, bw, _ in HW_ROWS:
    print(f'  {name:<26}{transfer_ms(bytes_per_expert, bw):>11.2f} ms   有效带宽 {bw:>7.1f} GB/s')

hbm_ms = transfer_ms(bytes_per_expert, HBM_BW)
pcie_ms = transfer_ms(bytes_per_expert, PCIE_BW)
nvme_ms = transfer_ms(bytes_per_expert, NVME_BW)

# 若一个 token 的 64 个专家全部流式搬入（无缓存），要多久？
stream_ms_per_token = N_LAYERS * TOP_K * pcie_ms
print(f'\n最坏情况：不做缓存，每 token 流式搬 {N_LAYERS * TOP_K} 个专家')
print(f'  = {N_LAYERS * TOP_K} × {pcie_ms:.2f} ms = {stream_ms_per_token:.1f} ms/token '
      f'→ 理论吞吐上限 {1000 / stream_ms_per_token:.2f} token/s')
print(f'对照：同样的专家若已在 HBM，搬运 ≈ {transfer_ms(bytes_per_expert, HBM_BW):.2f} ms（可忽略）')

assert nvme_ms > pcie_ms > hbm_ms, (nvme_ms, pcie_ms, hbm_ms)
assert 3.4 <= nvme_ms / pcie_ms <= 3.8, nvme_ms / pcie_ms          # 25/7 ≈ 3.57
assert 850.0 <= stream_ms_per_token <= 950.0, stream_ms_per_token   # ≈901.8 ms
print('\n✓ 通道断言通过：PCIe 单专家≈14 ms，全流式≈902 ms/token（这就是必须缓存的原因）')


## 3️⃣ 机制图

![](images/11_expert_paging.svg)

**图 1**：专家分页的三层结构——左：逻辑页表把 `(layer, expert)` 映射到 GPU 槽位，`-1` 表示缺页；中：槽位池容量固定（slots），满了按 LRU 淘汰最久未用的槽位；右：CPU/DRAM 存全量副本、NVMe 存冷专家；底部黄色车道是唯一的数据通道 —— PCIe 4.0 ×16 ≈ 25 GB/s，单专家 352 MB ≈ 14.1 ms。

一次访存的完整状态机（下一个 code cell 会把它打印出来）：

```text
token 路由 → (layer l, expert e)
                 │
                 ├─ page_table[l][e] != -1 ──命中──► 直接拿权重算（0 搬运）
                 │
                 └─ == -1 ──缺页──► 有空格子?
                                        ├─ 有 → 搬入空格子
                                        └─ 无 → 选 victim(LRU/LFU/FIFO)
                                                 → 旧页表项置 -1 ★最容易漏★
                                                 → 搬入 → 写新页表项
```

四个要点：

1. **页表是唯一的真相源**：槽位内容可以随意替换，只要页表同步；
2. **命中判定在路由之后**：所以路由（token→专家）必须确定，才能查表；
3. **搬运是同步阻塞的**（朴素实现），SM 在这 14 ms 里没有活干 → 这叫 **stall**；
4. **跨请求共享**：专家的身份是 `(layer, expert)`，与请求无关——这是与 KV 页最本质的区别。


In [ ]:
def print_flow():
    art = r'''
  ┌──────────────┐   page_table[l][e] != -1   ┌────────────────────┐
  │ token 路由    │ ─────────────────────────► │ 命中：直接读权重算  │
  │ (l, e)       │                            └────────────────────┘
  └──────┬───────┘
         │ == -1（缺页 fault）
         ▼
  ┌──────────────┐   有空槽位?  否   ┌──────────────────────────────┐
  │ 查 DRAM 副本  │ ───────────────► │ 选 victim（LRU/LFU/FIFO）     │
  └──────┬───────┘                  │ → 旧页表项置 -1  ★必须做★      │
         │ 是                        └───────────────┬──────────────┘
         ▼                                          │
  ┌──────────────┐                                    │
  │ 搬 352 MB     │ ◄──────────────────────────────────┘
  │ ≈14.1 ms      │
  └──────┬───────┘
         ▼
  ┌──────────────┐
  │ 写页表 + 计数  │
  └──────────────┘
'''
    print(art)


print_flow()

# 用一个极小的「手抄版」页表演示状态变化（还没引入 ExpertPager，先看语义）
page_table = {(0, 0): 0, (0, 1): 1, (0, 3): 2, (1, 1): 3}
slots = {0: (0, 0), 1: (0, 1), 2: (0, 3), 3: (1, 1)}

def show(state_note):
    print(f'\n[{state_note}]')
    print('  slots :', {k: v for k, v in sorted(slots.items())})
    print('  页表   :', {k: v for k, v in sorted(page_table.items())})

show('初始：4 个专家驻留')
print('\n访问 (2, 5)：页表没有 → 缺页；假设淘汰 slot 2（原本是 (0,3)）')
old_layer, old_expert = slots[2]
del page_table[(old_layer, old_expert)]        # ★ 一条都不能少
page_table[(2, 5)] = 2
slots[2] = (2, 5)
show('淘汰后：slot 2 已换成 (2,5)，旧页表项已删除')

assert page_table[(2, 5)] == 2 and (0, 3) not in page_table
print('\n✓ 命中判定只看页表：命中率 = 页表有效次数 / 总访问次数')


## 4️⃣ 手写实现

接口契约（面试时按这个顺序讲）：

| 成员 | 语义 |
|---|---|
| `page_table[layer][expert]` | 逻辑页 → 槽位号，`-1` 表示不在显存 |
| `slot_page[slot]` | 槽位 → `(layer, expert)`，反向索引，淘汰时用来失效旧页表项 |
| `policy` | `'LRU'` / `'LFU'` / `'FIFO'` 三选一 |
| `access(layer, expert)` | 返回 `True`=命中；`False`=缺页（内部完成搬运记账） |
| `hits / misses / bytes_moved / stall_ms()` | 统计口径：命中率、搬运字节、停顿时间 = 字节 / 带宽 |
| `reset_stats()` | 只清统计不清缓存，用于区分「冷启动」与「稳态」 |


In [ ]:
class ExpertPager:
    '''把 (layer, expert) 逻辑页映射到 GPU 固定槽位；容量不足时按策略淘汰。

    policy : 'LRU'（最久未用）/ 'LFU'（使用次数最少）/ 'FIFO'（最早装入）
    记账   : 每次缺页记一次搬运 bytes_per_expert，stall = bytes / bw
    '''

    def __init__(self, n_layers, n_experts, n_slots, policy='LRU',
                 bytes_per_expert=352_321_536, bw_gbps=25.0):
        if policy not in ('LRU', 'LFU', 'FIFO'):
            raise ValueError(f'unknown policy: {policy}')
        self.n_layers, self.n_experts = int(n_layers), int(n_experts)
        self.n_slots = int(n_slots)
        self.policy = policy
        self.bytes_per_expert = float(bytes_per_expert)
        self.bw_gbps = float(bw_gbps)

        self.page_table = [[-1] * self.n_experts for _ in range(self.n_layers)]
        self.slot_page = [(-1, -1)] * self.n_slots      # 槽位 → (layer, expert)
        self._stamp = [0] * self.n_slots                # LRU 最近使用时刻 / FIFO 装入时刻
        self._freq = [0] * self.n_slots                 # LFU 使用次数
        self._clock = 0
        self.reset_stats()

    # ---------------- 统计 ----------------
    def reset_stats(self):
        self.hits = 0
        self.misses = 0
        self.bytes_moved = 0.0
        self.per_layer_miss = [0] * self.n_layers

    def n_access(self):
        return self.hits + self.misses

    def hit_rate(self):
        return self.hits / self.n_access() if self.n_access() else 0.0

    def stall_ms(self):
        '''搬运字节 / 带宽 → 累计 stall（朴素同步实现，无 overlap）'''
        return self.bytes_moved / GB / self.bw_gbps * 1e3

    def resident_pages(self):
        return sum(1 for s in self.slot_page if s[0] != -1)

    # ---------------- 内部 ----------------
    def _free_slot(self):
        for s in range(self.n_slots):
            if self.slot_page[s][0] == -1:
                return s
        return -1

    def _victim(self):
        used = [s for s in range(self.n_slots) if self.slot_page[s][0] != -1]
        if not used:
            return -1
        if self.policy in ('LRU', 'FIFO'):
            return min(used, key=lambda s: self._stamp[s])
        # LFU：次数最少优先；次数相同则更久未用者优先（避免抖动）
        return min(used, key=lambda s: (self._freq[s], self._stamp[s]))

    def _evict(self, slot):
        '''淘汰：把旧页表项置 -1。单独成方法，方便后面「常见坑」演示漏写它的后果'''
        old_layer, old_expert = self.slot_page[slot]
        if old_layer != -1:
            self.page_table[old_layer][old_expert] = -1

    # ---------------- 主路径 ----------------
    def access(self, layer, expert):
        self._clock += 1
        slot = self.page_table[layer][expert]
        if slot != -1:                                  # 命中：零搬运
            self.hits += 1
            if self.policy == 'LRU':
                self._stamp[slot] = self._clock
            self._freq[slot] += 1
            return True

        # ---------- 缺页 ----------
        self.misses += 1
        self.per_layer_miss[layer] += 1
        self.bytes_moved += self.bytes_per_expert        # 只有缺页才搬字节
        slot = self._free_slot()
        if slot == -1:
            slot = self._victim()
            self._evict(slot)
        self.slot_page[slot] = (layer, expert)
        self.page_table[layer][expert] = slot
        self._stamp[slot] = self._clock
        self._freq[slot] = 1
        return False


p_demo = ExpertPager(SIM_LAYERS, N_EXPERTS, n_slots=8, policy='LRU',
                     bytes_per_expert=bytes_per_expert, bw_gbps=PCIE_BW)
for _ in range(40):
    p_demo.access(0, 7)          # 第 1 次缺页（冷启动），其后 39 次命中
p_demo.access(1, 3)              # 再缺页一次
print(f'演示：41 次访问 → 命中 {p_demo.hits} / 缺页 {p_demo.misses}，'
      f'命中率 {p_demo.hit_rate():.3f}')
print(f'      搬运 {p_demo.bytes_moved / MB:.1f} MB，stall {p_demo.stall_ms():.2f} ms '
      f'（= 2 × {transfer_ms(bytes_per_expert, PCIE_BW):.2f} ms）')
assert p_demo.hits == 39 and p_demo.misses == 2
assert abs(p_demo.stall_ms() - 2 * transfer_ms(bytes_per_expert, PCIE_BW)) < 1e-9


### 4.1 合成路由轨迹：专家热度是 Zipf 的

真实 MoE 的路由统计规律：**少数专家包揽大部分 token**（负载不均），近似 Zipf 分布 $p_r \propto 1/r^{\alpha}$，$r$ 是热度排名。

- $\alpha$ 越大越倾斜（少数王牌专家）；$\alpha \to 0$ 退化为均匀路由；
- 每层独立随机置换排名（不同层的专家分工不同）；
- 每个 token 每层取 `top_k` 个**不重复**专家 —— 这就是一条 `(layer, expert)` 访问序列。

**关键细节：路由必须带「token 间相关性」。** 如果每个 token 都独立重抽样，缓存命中率会一律趋近 0（现实中不会这样）。真实路由器对连续 token 打分高度相关（相邻 token 的专家集合重叠常在 60%~80%），所以这里用

$$ \text{score}_{l,e} = \log p_{\text{rank}(l,e)} + \sigma\cdot \text{Gumbel}(0,1) $$

$\sigma\to 0$ → 路由几乎确定（缓存极友好）；$\sigma=1$ → 退化为独立采样（缓存必然抖动）。本节取 $\sigma=0.5$ 居中。


In [ ]:
def simulate_routing(tokens, layers, top_k, zipf_alpha, sigma=0.5,
                     n_experts=N_EXPERTS, seed=2024):
    '''生成合成 MoE 路由轨迹（带 token 间相关性）。

    打分模型：score[l][e] = log p_rank[l][e] + sigma · Gumbel(0,1)
      · p_rank 是每层按 Zipf(alpha) 排好的专家偏好（每层独立置换，模拟不同的专家分工）；
      · sigma 控制抖动：sigma→0 时路由几乎确定（相邻 token 高度重叠，缓存友好），
        sigma=1 退化独立采样（Plackett–Luce），相邻 token 无相关性 → 缓存必然被击穿。
      真实路由器的相邻 token overlap 通常在 60%~80%，取 sigma=0.5 居中。

    返回 list[(layer, expert)]，长度 = tokens × layers × top_k。
    '''
    rng = np.random.default_rng(seed)
    rank = np.arange(1, n_experts + 1, dtype=float)
    p_rank = rank ** (-zipf_alpha)
    p_rank = p_rank / p_rank.sum()
    perms = [rng.permutation(n_experts) for _ in range(layers)]
    base = np.array([np.log(p_rank[perms[l]]) for l in range(layers)])

    trace = []
    for _ in range(tokens):
        score = base + sigma * rng.gumbel(size=(layers, n_experts))
        picks = np.argsort(-score, axis=1)[:, :top_k]
        for l in range(layers):
            for e in picks[l]:
                trace.append((l, int(e)))
    return trace


TRACE = simulate_routing(tokens=256, layers=SIM_LAYERS, top_k=TOP_K,
                         zipf_alpha=1.0, seed=11)
print(f'轨迹长度 {len(TRACE)} = 256 token × {SIM_LAYERS} 层 × {TOP_K} 专家')
print(f'覆盖的唯一页数 {len(set(TRACE))} / {SIM_PAGES}（缩比模型的全部专家页）')

# 热度检查：第 0 层各专家被访问的比例 vs Zipf 偏好
layer0 = [e for l, e in TRACE if l == 0]
share = np.bincount(layer0, minlength=N_EXPERTS) / len(layer0)
p_rank1 = 1.0 / np.arange(1, N_EXPERTS + 1)
p_rank1 = p_rank1 / p_rank1.sum()
print('\n第 0 层专家访问占比（降序）：', np.round(np.sort(share)[::-1], 3))
print('Zipf(α=1) 偏好（降序）      ：', np.round(np.sort(p_rank1)[::-1], 3))
print(f'最热专家实测占比 {share.max():.3f} vs 纯采样偏好 {p_rank1.max():.3f}'
      '（σ<1 让路由比纯采样更确定，所以实测更集中）')

assert len(TRACE) == 256 * SIM_LAYERS * TOP_K, len(TRACE)
assert len(set(TRACE)) == SIM_PAGES, len(set(TRACE))
assert 0.25 <= share.max() <= 0.75, share.max()


### 4.2 与 PagedAttention 的对照表（面试必答）

| 维度 | **KV 页（PagedAttention）** | **专家页（Expert Paging）** |
|---|---|---|
| 扩展单位 | 序列块（如 16 token / block） | 一个专家权重 `(layer, expert)` |
| Key 的构成 | 请求 id + 逻辑块号 | `(layer, expert)` 下标 |
| 生命周期 | 随请求创建、随序列增长、请求结束即释放 | 跨请求常驻或反复换入换出，**不随请求结束失效** |
| 是否独占 | **按请求独占**（同前缀可 COW 共享） | **全局共享**（所有 token / 请求复用同一份只读权重） |
| 大小 | 与序列长度成正比（KV 头 × 层数 × 块） | 固定 3·d·d_ff×2 B ≈ 352 MB |
| 写回 | 可能脏、需要写回 | **只读**，淘汰可直接丢弃 |
| 淘汰目标 | 请求结束 → 直接归还池子 | 按热度（LRU/LFU），miss 代价是 **14.1 ms 停顿** |
| 优化目标 | 消灭碎片、提高并发 | 提高命中率、隐藏 PCIe 搬运 |
| 关键指标 | 峰值显存占用 / 并发数 | hit rate / stall 时间 / 有效吞吐 |


In [ ]:
# 用数字说明「共享 vs 独占」（这张表最容易被追问）
requests = ['req-A', 'req-B', 'req-C']

# KV 页：每个请求各自持有一份块，不能互相顶替
kv_blocks = {r: [f'{r}-blk{i}' for i in range(3)] for r in requests}
kv_total = sum(len(v) for v in kv_blocks.values())

# 专家页：三个请求在同一层命中同样的原子（只读权重），只需一份
needed = {(0, 7), (1, 2)}
expert_if_private = sum(len(needed) for _ in requests)   # 若按请求独占
expert_if_shared = len(set().union(*[needed for _ in requests]))

print('3 个请求、每个 3 个 KV 块 / 每个请求用到 2 个专家：')
print(f'  KV 页占用块数        : {kv_total}（每请求独占，无法共享）')
print(f'  专家页占用槽位数     : 独占 {expert_if_private}  vs  共享 {expert_if_shared}')
print(f'  节省                 : {1 - expert_if_shared / expert_if_private:.0%} 的专家显存')

assert kv_total == 9
assert expert_if_private == 6 and expert_if_shared == 2
print('\n✓ 这就是「专家页跨请求共享」的定量含义：并发越高，摊薄越明显')


## 5️⃣ 数值实验

实验矩阵（全部固定 seed，可复现）：

| 实验 | 变量 | 观测量 | 断言 |
|---|---|---|---|
| E1 | slots ∈ {4,8,16,32,64} × policy ∈ {LRU, LFU, FIFO} | 稳态命中率曲线 | slots≥工作集 → miss=0；LRU ≥ FIFO |
| E2 | 同上 | 搬运字节 / stall | 字节 = miss × 单专家字节（误差 <1e-9） |
| E3 | Zipf α ∈ {0.6, 1.0, 1.4} | 热度分布 + 命中率 | α 越大命中率越高 |

**为什么用缩比模型跑 slots 网格？** slots ∈ {4..64} 直接套在 32 层 Mixtral 上，等于「每层平均 0.1~2 个专家」——绝大部分配置恒在抖动区（命中率 0），看不出任何趋势。所以数值实验改用 **4 层 × 8 专家 = 32 页** 的等价缩比配置：单专家字节数、PCIe 带宽、Zipf 路由形状都不变，只是把「专家数」缩到能在 4~64 个槽位里观察到容量效应；真实 32 层时按「每层几个专家」等比放大即可（每层 2 个 ≈ 64 槽位）。

⚠️ **冷启动 ≠ 稳态**：第一轮 epoch 必然全是强制性缺失（miss），所以所有命中率都在「热身一轮 → 清空统计 → 再跑一轮」之后测量——这点不说清楚，面试官会认为你在刷指标。


In [ ]:
def run_trace(pager, trace):
    '''按轨迹访问一遍，返回同一个 pager（便于链式统计）'''
    for layer, expert in trace:
        pager.access(layer, expert)
    return pager


def steady_state(pager, trace):
    '''热身一轮丢掉冷启动缺失，再清零统计跑第二轮 → 稳态指标'''
    run_trace(pager, trace)
    pager.reset_stats()
    run_trace(pager, trace)
    return pager


SLOT_GRID = [4, 8, 16, 32, 64]
POLICIES = ('LRU', 'LFU', 'FIFO')
scan = {}
for slots in SLOT_GRID:
    for pol in POLICIES:
        p = ExpertPager(SIM_LAYERS, N_EXPERTS, slots, pol, bytes_per_expert, PCIE_BW)
        steady_state(p, TRACE)
        scan[(slots, pol)] = p

print(f'稳态命中率（轨迹：256 token × {SIM_LAYERS} 层 × top-2 = {len(TRACE)} 次访问，{SIM_PAGES} 页）')
print(f'场景：slots 个槽位在 {SIM_LAYERS} 层之间共享 → 每层平均只有 slots/{SIM_LAYERS} 个专家可驻留')
print('      注意 slots=4 恰好是「每层 1 个」，而每 token 每层要走 2 个专家 → 必然抖动（命中率 0）')
print('-' * 62)
print(f'{"slots":>6}{"每层平均":>10}{"LRU":>10}{"LFU":>10}{"FIFO":>10}')
for slots in SLOT_GRID:
    row = [scan[(slots, pol)].hit_rate() for pol in POLICIES]
    print(f'{slots:>6}{slots / SIM_LAYERS:>10.2f}'
          f'{row[0]:>10.3f}{row[1]:>10.3f}{row[2]:>10.3f}')
print('-' * 62)

# 命中率随 slots 单调不减（容量更大不可能更差）
for pol in POLICIES:
    hrs = [scan[(s, pol)].hit_rate() for s in SLOT_GRID]
    assert all(b >= a - 1e-12 for a, b in zip(hrs, hrs[1:])), (pol, hrs)
print('✓ 三种策略的命中率都随 slots 单调不减')


In [ ]:
plt.figure(figsize=(8.2, 3.8))
for pol, marker in zip(POLICIES, ('o-', 's--', '^:')):
    hrs = [scan[(s, pol)].hit_rate() for s in SLOT_GRID]
    plt.plot(SLOT_GRID, hrs, marker, label=f'{pol}')
plt.xscale('log', base=2)
plt.xticks(SLOT_GRID, [str(s) for s in SLOT_GRID])
plt.xlabel(f'GPU 槽位数 slots（{SIM_LAYERS} 层 × {N_EXPERTS} 专家 = {SIM_PAGES} 页）')
plt.ylabel('稳态命中率')
plt.title('专家缓存命中率 vs 槽位容量（Zipf α=1，top-2）')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print('读图：')
print('  1) 4→8 是拐点（0.00 → 0.53）：每层 1 个槽位时路由每步都在换专家，LRU 被完全击穿；')
print('  2) 8→16 仍在涨（0.53 → 0.83），因为每层 8 个专家里真正热的只有 2~4 个；')
print('  3) 32 之后饱和：32 页全驻留，miss = 0 —— 容量换停顿的极限就是这样。')
print('  4) LFU 在两个中间档优于 LRU（0.61 vs 0.53）：路由带噪声时频次比"最近用过"更稳；')
print('     但 FIFO 一律最差，说明"装入顺序"与热度无关时会白扔热页。')


In [ ]:
# ---------------- 断言集：容量、策略、字节守恒 ----------------
# ① 槽位 ≥ 轨迹里出现的唯一页数 → 稳态 miss = 0（只剩冷启动的强制性缺失）
distinct_pages = len(set(TRACE))
p_big = ExpertPager(SIM_LAYERS, N_EXPERTS, distinct_pages, 'LRU', bytes_per_expert, PCIE_BW)
steady_state(p_big, TRACE)
print(f'① slots = {distinct_pages}（= 唯一页数）：稳态 miss = {p_big.misses}，'
      f'命中率 = {p_big.hit_rate():.4f}')
assert p_big.misses == 0 and p_big.hit_rate() > 0.999999

# ② 同一轨迹下 LRU 命中率 ≥ FIFO（Zipf 局部性 → 时间局部性更重要）
for slots in SLOT_GRID:
    h_lru = scan[(slots, 'LRU')].hit_rate()
    h_bit = scan[(slots, 'FIFO')].hit_rate()
    print(f'② slots={slots:>3}: LRU {h_lru:.4f} vs FIFO {h_bit:.4f}  '
          f'{"LRU 更优" if h_lru > h_bit else "持平/更差"}')
    assert h_lru >= h_bit - 1e-12, (slots, h_lru, h_bit)

# ③ 搬运字节 = miss 数 × 单专家字节（守恒，误差 < 1e-9 B）
for slots in SLOT_GRID:
    p = scan[(slots, 'LRU')]
    expect = p.misses * bytes_per_expert
    err = abs(p.bytes_moved - expect)
    assert err < 1e-9, (slots, err)
    assert abs(p.stall_ms() - p.misses * pcie_ms) < 1e-9 * max(1.0, p.stall_ms()), (slots, p.stall_ms())
print(f'③ 搬运字节守恒：{len(SLOT_GRID)} 组配置误差均 < 1e-9 B；'
      f'stall 与 miss × {pcie_ms:.2f} ms 一致')

# ④ 缓存实际驻留不超过容量（别把「页表有效」当「槽位够用」）
for slots in SLOT_GRID:
    p = scan[(slots, 'LRU')]
    assert p.resident_pages() <= slots
print('④ 驻留页数 ≤ slots：全部配置通过')

print('\n✓ 断言 ①②③④ 全部通过')


In [ ]:
# ---------------- E3：Zipf α 对命中率的影响 ----------------
SUBSET = [8, 16, 32, 64]
ALPHAS = [0.6, 1.0, 1.4]
alpha_scan = {}
alpha_trace = {}

for alpha in ALPHAS:
    tr = simulate_routing(256, SIM_LAYERS, TOP_K, alpha, seed=11)
    alpha_trace[alpha] = tr
    for slots in SUBSET:
        p = ExpertPager(SIM_LAYERS, N_EXPERTS, slots, 'LRU', bytes_per_expert, PCIE_BW)
        steady_state(p, tr)
        alpha_scan[(alpha, slots)] = p.hit_rate()

print('稳态命中率（LRU）随 Zipf 倾斜度变化：')
print(f'{"α":>5}' + ''.join(f'{("slots=" + str(s)):>12}' for s in SUBSET))
for alpha in ALPHAS:
    print(f'{alpha:>5.1f}' + ''.join(f'{alpha_scan[(alpha, s)]:>12.3f}' for s in SUBSET))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
rank = np.arange(1, N_EXPERTS + 1)
for alpha in ALPHAS:
    p = rank ** (-alpha)
    axes[0].bar(rank + (alpha - 1.0) * 0.22, p / p.sum(), width=0.22,
                label=f'α={alpha}')
axes[0].set_xlabel('专家热度排名 r')
axes[0].set_ylabel('被路由概率 $p_r$')
axes[0].set_title('Zipf 热度分布：α 越大越集中')
axes[0].legend()
axes[0].grid(alpha=0.25, axis='y')

for alpha in ALPHAS:
    axes[1].plot(SUBSET, [alpha_scan[(alpha, s)] for s in SUBSET], 'o-', label=f'α={alpha}')
axes[1].set_xscale('log', base=2)
axes[1].set_xticks(SUBSET)
axes[1].set_xticklabels([str(s) for s in SUBSET])
axes[1].set_xlabel('GPU 槽位数 slots')
axes[1].set_ylabel('稳态命中率')
axes[1].set_title('热度越集中，缓存越有效')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 断言：容量≥2 槽/层时，α 越大命中率越高（热专家更容易被缓存住）
# slots=8 时只剩 1 槽/层以内，命中率由抖动主导，不参与单调性断言
for slots in [s for s in SUBSET if s >= 8]:
    h = [alpha_scan[(a, slots)] for a in ALPHAS]
    assert h[2] >= h[1] - 1e-12 and h[1] >= h[0] - 1e-12, (slots, h)
mean_h = [float(np.mean([alpha_scan[(a, s)] for s in SUBSET])) for a in ALPHAS]
assert mean_h[2] >= mean_h[1] - 1e-12 and mean_h[1] >= mean_h[0] - 1e-12, mean_h
print(f'✓ α 单调性断言通过：平均命中率 {mean_h[0]:.3f} → {mean_h[1]:.3f} → {mean_h[2]:.3f}')
assert len(alpha_trace) == 3


In [ ]:
# ---------------- stall 账：命中率 → 每 token 停顿 → 隐含吞吐 ----------------
print('把命中率换算成在线服务最关心的两个数（稳态，LRU）：')
print('-' * 84)
print(f'{"slots":>6}{"命中率":>9}{"每 token miss 数":>16}{"每 token stall(ms)":>20}{"停顿隐含吞吐(tok/s)":>22}')
stall_rows = []
for slots in SLOT_GRID:
    p = scan[(slots, 'LRU')]
    n = p.n_access()
    miss_rate = p.misses / n
    miss_per_token = miss_rate * N_LAYERS * TOP_K   # 折算回 32 层 Mixtral
    stall_ms = miss_per_token * pcie_ms
    stall_rows.append((slots, p.hit_rate(), miss_per_token, stall_ms))
    print(f'{slots:>6}{p.hit_rate():>9.3f}{miss_per_token:>16.2f}'
          f'{stall_ms:>20.2f}{1000 / stall_ms if stall_ms > 0 else float("inf"):>22.2f}')
print('-' * 84)
print('注意：这是「朴素同步搬运」的下界估计，工程实现靠预取/多流 overlap 把 stall 藏起来。')

assert all(b[3] <= a[3] + 1e-12 for a, b in zip(stall_rows, stall_rows[1:]))
assert stall_rows[0][3] > 300.0, stall_rows[0][3]
print('\n✓ slack 断言通过：slots 越大 stall 越小；slots=4 时每 token 停顿 > 300 ms')


## 6️⃣ 工程对照（定性对照，不编造具体数字）

四条主流路线，面试时讲清「解决什么、代价是什么」即可（具体实现见各自官方仓库）：

| 方案 | 核心机制 | 解决什么 | 代价 / 局限 |
|---|---|---|---|
| **Mixtral-offloading**（工程实现，详见官方仓库） | offload 专家到 CPU/DRAM，**LRU 缓存 + 投机预取**（用上一 token 的路由预测下一 token 的专家） | 单卡跑 8x7B；把 14 ms 搬运藏进计算 | 预取错就白搬一份；依赖路由的时序相关性 |
| **MoE-Infinity**（工程实现，详见官方仓库） | **激活感知（activation-aware）专家缓存**：按「即将被哪些 token 激活」预取，而不是只看历史频次 | 长序列 / 大 batch 下的缓存抖动 | 需要拿到下一批激活（调度耦合）；实现复杂 |
| **hot expert replication** | 把最热的专家复制多份放到不同卡/不同槽位 | 负载均衡：消除「王牌专家」所在卡的排队 | 复制占容量 → 稳态命中率下降（第 8️⃣ 节有量化） |
| **HBM → DRAM → NVMe 三级存储** | 显存放热页、主机 DRAM 放全量、NVMe 放冷专家/多模型 | 超出单机内存的模型库；快速换任务 | 带宽比 25:7≈3.6×，两跳延迟 ≈4.6×；需要容量规划 |

三条工程经验（背下来）：

1. **预取比淘汰算法更重要**：LRU 换 FIFO 只影响命中率几个点，而一次预取命中能直接省掉 14 ms；
2. **搬运要能和计算 overlap**：多 CUDA stream + 分层搬运（先搬 gate/up，算的同时搬 down），才有接近零停顿的可能；
3. **专家页只读** ⇒ 淘汰时不写回、可直接覆盖 ⇒ 实现比 KV 页简单，但**必须**同步失效页表（否则双租）。


In [ ]:
# 工程方案对照表（定性，仅列出机制与代价，不写未经核实的性能数字）
eng = [
    ('Mixtral-offloading', 'LRU 缓存 + 上一 token 路由做投机预取',
     '单卡跑 8x7B；搬运藏进计算', '预取失败白搬；依赖路由时序相关性'),
    ('MoE-Infinity', '激活感知专家缓存（按即将激活的 token 预取）',
     '大 batch / 长序列下减少抖动', '与调度耦合；实现复杂'),
    ('hot expert replication', '最热专家复制多份到多卡/多槽位',
     '消除王牌专家的排队瓶颈', '复制占容量 → 稳态命中率下降'),
    ('三级存储 HBM→DRAM→NVMe', '热页在显存、全量在 DRAM、冷专家在 NVMe',
     '模型库超单机内存；快速换任务', '带宽比 3.6×，两跳延迟 ≈4.6×'),
]
print(pad('方案', 26) + pad('机制', 44) + pad('收益', 32) + '代价')
print('-' * 132)
for name, mech, gain, cost in eng:
    print(pad(name, 26) + pad(mech, 44) + pad(gain, 32) + cost)
print('-' * 132)
assert len(eng) == 4
print('✓ 四条工程路线：前两条是「缓存 + 预取」，后两条是「复制」与「分层」')


## 7️⃣ 三级存储与冷加载预算

显存不够时，专家权重分布在三层介质上，**每层都便宜一个量级、也慢一个量级**：

```text
HBM（GPU）        几十 GB   算子直接读          ← 命中即 0 停顿
  ↕ PCIe 4.0      25 GB/s   一次 14.1 ms/专家
DRAM（主机）     几百 GB   全量副本（90.2 GB）  ← 缺页的落点
  ↕ PCIe/NVMe     7 GB/s    一次 50.3 ms/专家
NVMe（SSD）      TB 级     冷专家 / 多模型权重库
```

两个常用的预算问题：

1. **冷启动**：从 DRAM 把全部专家搬进显存 ≈ $90.2/25 = 3.61$ s（真实系统会懒加载，只搬用到的）；
2. **从 NVMe 起步**：≈ $90.2/7 = 12.9$ s —— 所以「换模型 = 用户感知的秒级卡顿」，工程上靠常驻/pre-warm 解决。


In [ ]:
t_all_dram_s = mem_all_experts / GB / PCIE_BW
t_all_nvme_s = mem_all_experts / GB / NVME_BW
t_expert_nvme_ms = transfer_ms(bytes_per_expert, NVME_BW)
t_expert_nvme_to_hbm_ms = t_expert_nvme_ms + pcie_ms      # NVMe → DRAM → HBM 两跳

print('冷加载预算（全量 90.19 GB）：')
print(f'  DRAM → HBM : {t_all_dram_s:>7.2f} s')
print(f'  NVMe → HBM : {t_all_nvme_s:>7.2f} s')
print(f'  单专家 NVMe → DRAM → HBM : {t_expert_nvme_to_hbm_ms:>7.2f} ms')
print(f'  单专家 DRAM → HBM        : {pcie_ms:>7.2f} ms')
print(f'  换层倍数                 : {t_expert_nvme_to_hbm_ms / pcie_ms:>7.2f}×')

# 若只允许 8 个槽位（≈2.8 GB）却要服务 256 页 → 必须靠命中率
slots_budget = 8
print(f'\n8 个槽位 = {slots_budget * bytes_per_expert / GB:.1f} GB '
      f'= 全量专家的 {slots_budget / (N_LAYERS * N_EXPERTS):.1%}')

assert 3.4 <= t_all_dram_s <= 3.9, t_all_dram_s
assert 12.0 <= t_all_nvme_s <= 14.0, t_all_nvme_s
assert 62.0 <= t_expert_nvme_to_hbm_ms <= 68.0, t_expert_nvme_to_hbm_ms
assert 2.7 <= slots_budget * bytes_per_expert / GB <= 2.9
print('\n✓ 三级存储断言通过：NVMe 起步 ≈12.9 s，单专家两跳 ≈64 ms')


## 8️⃣ 热点专家复制（hot expert replication）：容量与负载的取舍

问题：Zipf 路由会让**同一个专家**被大量 token 同时需要 → 该专家所在卡/槽位成为排队瓶颈。

解法：把热专家复制 $rep$ 份，分摊并发。代价是**复制品占用了珍贵的槽位**：

- 有效「不同页」容量从 $C$ 降到 $C/rep$；
- 稳态命中率 ≈ Zipf 覆盖率 $\sum_{r\le C/rep} p_r$，随 $rep$ 单调下降。

结论（下面用数字验证）：**复制只值得用在「突发 / 排队抖动」场景**，稳态命中率一定变差；所以真实系统要么只在少数王牌专家上复制，要么配激活感知预热来抵消。


In [ ]:
def zipf_coverage(capacity, n_pages, alpha):
    '''容量为 capacity（可为小数）时，按 Zipf(alpha) 排序取前 capacity 页的覆盖概率'''
    p = np.arange(1, n_pages + 1, dtype=float) ** (-alpha)
    p = np.sort(p)[::-1]
    p = p / p.sum()
    k = int(np.floor(capacity))
    cover = p[:k].sum()
    if k < n_pages:
        cover += (capacity - k) * p[k]          # 小数容量按比例计入下一页
    return float(cover)


print(f'单层视角：8 个专家、Zipf α=1；把 slots 平均摊到 {SIM_LAYERS} 层后，每个复制系数下的覆盖率')
print(f'{"slots":>6}{"每层容量":>10}{"rep=1":>10}{"rep=2":>10}{"rep=3":>10}')
repl = {}
for slots in SLOT_GRID:
    per_layer = slots / SIM_LAYERS
    vals = []
    for rep in (1, 2, 3):
        c = zipf_coverage(per_layer / rep, N_EXPERTS, 1.0)
        repl[(slots, rep)] = c
        vals.append(c)
    print(f'{slots:>6}{per_layer:>10.2f}{vals[0]:>10.3f}{vals[1]:>10.3f}{vals[2]:>10.3f}')

plt.figure(figsize=(7.6, 3.6))
for rep, mk in zip((1, 2, 3), ('o-', 's--', '^:')):
    plt.plot(SLOT_GRID, [repl[(s, rep)] for s in SLOT_GRID], mk, label=f'复制 {rep} 份')
plt.xscale('log', base=2)
plt.xticks(SLOT_GRID, [str(s) for s in SLOT_GRID])
plt.xlabel('GPU 槽位数 slots（复制品也占槽位）')
plt.ylabel('稳态覆盖率（≈命中率）')
plt.title('热点复制：负载均衡换命中率，容量越紧代价越大')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

for slots in SLOT_GRID:
    c1, c2, c3 = (repl[(slots, r)] for r in (1, 2, 3))
    assert c1 >= c2 - 1e-12 and c2 >= c3 - 1e-12, (slots, c1, c2, c3)
print('✓ 断言通过：复制份数越多稳态覆盖率越低（复用原有槽位 → 有效容量下降）')
print('   推论：复制只该用在「被抢占/排队」的热点上，且必须配合预热，否则净收益为负。')


## 🎯 面试考点

1. **为什么 MoE 必须 offload？** 90.2 GB 专家权重 vs 80 GB 单卡；每 token 只激活 25%，密度太低不值得全驻留。
2. **专家页和 KV 页的区别？** 前者按 `(层, 下标)`、跨请求共享、只读、miss 代价是 14 ms 停顿；后者按请求、随序列增长、可能脏、目标是消灭碎片。
3. **一次缺页多贵？** 352.3 MB ÷ 25 GB/s ≈ 14.1 ms；若每 token 全流式搬 64 个 → 902 ms/token，所以必须命中。
4. **LRU 怎么实现？** 槽位维护最近使用时间戳，命中更新、淘汰取最小；LFU 用计数器（需处理老化），FIFO 只看装入时间。
5. **为什么 LRU 通常优于 FIFO？** Zipf 路由有强时间局部性；FIFO 会把仍热的页按装入顺序淘汰掉。
6. **淘汰时最容易出的 bug？** 忘了把旧页表项置 `-1` → 两个逻辑页指向同一槽位「双租」，命中判定失效。
7. **命中率怎么换算成吞吐？** 每 token 停顿 = miss 率 × 64 × 14.1 ms；停顿是串行的，所以吞吐上限 ≈ 1/停顿。
8. **预取为什么比换淘汰算法重要？** 淘汰算法的收益是「几个百分点的命中率」，预取命中是「直接省 14 ms」，量级不同。
9. **Mixtral-offloading 与 MoE-Infinity 的差别？** 前者用上一 token 路由做投机预取（历史频次）；后者用激活感知（即将激活的 token）预取，对大 batch / 长序列更稳。
10. **三级存储怎么规划？** 显存放热页（slots×352 MB）、DRAM 放全量（90.2 GB）、NVMe 放冷专家/多模型；带宽比 25:7≈3.6×、两跳延迟 ≈4.6×，换模型 ≈13 s 所以要 pre-warm。


### 🎯 追问集（答不上来说明只背了结论）

- **Q：slots 是不是越大越好？** 是单调不变差，但边际收益递减；且槽位挤占 KV cache 的空间——真实系统要在「专家命中率」和「并发上下文长度」之间做显存配额。
- **Q：为什么 top-2 的 MoE 每层要搬 2 个专家，而不是 1 个？** 因为两个专家都要参与计算；但两者可以流水线化（先搬 A 算了再搬 B），这就是 overlap 的空间。
- **Q：专家页能不能用 COW / 前缀共享？** 不需要——专家页天然全局共享且只读，比 COW 更简单；COW 是 KV 页为了省前缀重复才需要的。
- **Q：batch 里不同请求激活不同专家，缓存怎么算命中？** 命中率定义在「`(layer, expert)` 访问次数」上，而不是「请求数」上；batch 越大，同一专家被复用得越多 → 命中率越高。
- **Q：如果路由本身要做 all-to-all？** 那是 **专家并行（EP）** 的问题：权重按专家切到不同卡 → token 要走 NVLink/IB 过去算；本节讲的是**权重 offload（时间维度复用）**，两者解决的是容量 vs 带宽的不同瓶颈。


In [ ]:
# ---------------- 极端配置：slots=1 与 slots=工作集 ----------------
p_one = ExpertPager(SIM_LAYERS, N_EXPERTS, 1, 'LRU', bytes_per_expert, PCIE_BW)
steady_state(p_one, TRACE)
p_full = ExpertPager(SIM_LAYERS, N_EXPERTS, distinct_pages, 'LRU', bytes_per_expert, PCIE_BW)
steady_state(p_full, TRACE)

print(f'{"配置":<22}{"命中率":>10}{"每 epoch miss":>16}{"stall(ms)":>14}{"每 token stall(ms)":>20}')
for name, p in [('slots=1（只剩 1 页）', p_one),
                (f'slots={distinct_pages}（= 工作集）', p_full)]:
    per_tok = (p.misses / p.n_access()) * N_LAYERS * TOP_K * pcie_ms \
        if p.n_access() else 0.0
    print(pad(name, 26) + f'{p.hit_rate():>10.4f}{p.misses:>16d}'
          f'{p.stall_ms():>14.1f}{per_tok:>20.1f}')

assert p_one.hit_rate() < 0.35, p_one.hit_rate()
assert p_full.misses == 0 and p_full.stall_ms() == 0.0
assert p_one.stall_ms() > p_full.stall_ms()
print('\n✓ slots=1 几乎每步都在搬（命中率 < 0.35）；装满工作集则零停顿')


### ⚠️ 五个最容易栽的坑

1. **淘汰不失效旧页表** → 双租（两个逻辑页映射同一槽位），命中判定彻底失真；本节的 `BuggyPager` 是最小复现。
2. **顺序扫描击穿缓存**：路由如果按固定顺序轮流访问 $C+1$ 个页，LRU 命中率精确为 **0**（本节代码格里复现）——这是「专家缓存看起来一点用都没有」的经典原因。
3. **把冷启动算进命中率**：第一轮全是强制性缺失，报告指标必须给「稳态」定义。
4. **忘记 stall 是串行的**：把 14 ms 和计算时间简单相加还可以，但错误地按「并行」估算吞吐会高出数倍。
5. **专家页按请求分配**：把专家页当 KV 页管理（请求结束就释放）→ 并发下反复重搬，命中率被自己砍掉。


In [ ]:
# ---------------- 坑 1：淘汰不失效旧页表 ----------------
class BuggyPager(ExpertPager):
    '''故意不失效旧页表项的最小复现（面试题：这段代码错在哪）'''

    def _evict(self, slot):
        return  # ← bug：旧页表项留着，于是两个逻辑页指向同一个槽位


bug = BuggyPager(1, 9, 4, 'LRU', bytes_per_expert, PCIE_BW)
for e in range(4):
    bug.access(0, e)                 # 填满 4 个槽位
bug.access(0, 4)                     # 缺页 → 抢占 slot（旧页表项未失效）
dup = {}
for e, s in enumerate(bug.page_table[0]):
    if s != -1:
        dup.setdefault(s, []).append(e)
double_booked = {s: es for s, es in dup.items() if len(es) > 1}
print('页表（layer 0）：', bug.page_table[0])
print('同一槽位被多个专家指向：', double_booked)

stale_expert = list(double_booked.values())[0][0]
got = bug.access(0, stale_expert)     # 访问已被挤出去的专家
print(f'访问已被淘汰的专家 E{stale_expert}：buggy 版本返回命中 = {got}（实际数据已不在槽位里）')

good = ExpertPager(1, 9, 4, 'LRU', bytes_per_expert, PCIE_BW)
for e in range(4):
    good.access(0, e)
good.access(0, 4)
print('正确版本页表：', good.page_table[0])

assert len(double_booked) >= 1, '没能复现双租'
assert got is True
assert good.page_table[0][stale_expert] == -1
print('✓ 双租复现成功：正确实现里被淘汰专家的页表项必须是 -1')

# ---------------- 坑 2：顺序扫描击穿 LRU ----------------
cyclic = [(0, e) for e in range(9)] * 8      # 9 页循环，只有 8 个槽位
p_scan = ExpertPager(1, 9, 8, 'LRU', bytes_per_expert, PCIE_BW)
run_trace(p_scan, cyclic)
print(f'\n9 页循环 + 8 槽位：LRU 命中率 = {p_scan.hit_rate():.4f}（经典 LRU 抖动）')
assert p_scan.hit_rate() <= 0.02, p_scan.hit_rate()

# 稍微打乱顺序（模拟真实路由的局部性）后，同样的容量立刻有用
rng = np.random.default_rng(7)
local = []
for _ in range(72):
    local.append((0, int(rng.choice(9, p=np.array([6, 5, 4, 3, 2, 1, 1, 1, 1]) / 24.0))))
p_local = ExpertPager(1, 9, 8, 'LRU', bytes_per_expert, PCIE_BW)
run_trace(p_local, local)
print(f'同样 72 次访问但带热度偏斜：LRU 命中率 = {p_local.hit_rate():.4f}')
assert p_local.hit_rate() > p_scan.hit_rate()
print('✓ 命中率完全取决于路由的局部性，而不是缓存算法本身')


### 📖 术语速查（面试前默写一遍）

| 术语 | 一句话 |
|---|---|
| **Expert Paging** | 把 `(layer, expert)` 当逻辑页，按需映射到显存槽位 |
| **slot / 槽位池** | 显存里预留的固定大小权重槽，数量 = 缓存容量 |
| **page_table** | 逻辑页 → 槽位号；`-1` = 缺页 |
| **page fault / 缺页** | 页表无效 → 必须从下层搬运 |
| **LRU / LFU / FIFO** | 淘汰策略：最久未用 / 频次最低 / 最早装入 |
| **stall** | 因等待搬运而空转的时间，朴素实现下 = 字节 / 带宽 |
| **speculative prefetch** | 投机预取：用上一层/上一 token 的路由预测下一批专家 |
| **activation-aware caching** | 激活感知：按「即将被激活的 token」预取，而非历史频次 |
| **hot expert replication** | 热专家复制多份，换负载均衡、牺牲容量 |
| **offload / 三级存储** | HBM → DRAM → NVMe，容量与带宽各差一个量级 |
| **EP（专家并行）** | 权重按专家切到多卡，token 跨卡 all-to-all；与 offload 是两回事 |


In [ ]:
# ---------------- 一页纸结论（所有数字重新算一遍并与前面互相校验） ----------------
cold_load_bytes = N_LAYERS * N_EXPERTS * bytes_per_expert
assert abs(cold_load_bytes - mem_all_experts) < 1e-6 * mem_all_experts

best = scan[(16, 'LRU')]          # 4 槽/层 ≈ 每层热专家全驻留
best_per_tok = (best.misses / best.n_access()) * N_LAYERS * TOP_K * pcie_ms

print('=' * 74)
print('一页纸结论')
print('=' * 74)
print(f'单专家        : {params_per_expert / 1e6:.2f} M 参数 = {bytes_per_expert / MB:.1f} MB @BF16')
print(f'全部专家      : {params_all_experts / 1e9:.2f} G 参数 = {mem_all_experts / GB:.2f} GB')
print(f'PCIe 4.0 x16  : 单专家 {pcie_ms:.2f} ms / 全量 {mem_all_experts / GB / PCIE_BW:.2f} s')
print(f'无缓存最坏情形: {N_LAYERS * TOP_K} × {pcie_ms:.2f} ms = {stream_ms_per_token:.1f} ms/token')
print(f'16 槽位 + LRU : 命中率 {best.hit_rate():.3f} → 每 token 停顿 ≈ {best_per_tok:.1f} ms（折算 32 层）')
print(f'装满工作集    : 命中率 1.000 → 停顿 0 ms（容量换停顿）')
print('-' * 74)
print('一句话：专家分页用「容量 × 命中率」这一对变量，把 90 GB 的权重')
print('        塞进几十 GB 的显存，代价是每次缺页 14 ms 的同步停顿。')
print('=' * 74)

assert best.hit_rate() > 0.70          # 4 槽/层的 LRU 必须明显好于随机
assert best_per_tok < stream_ms_per_token
assert abs(mem_all_experts - 256 * bytes_per_expert) < 1.0
print('\n✓ 汇总断言通过：账本、缓存收益、最坏情形三者互相自洽')


## ✅ 自测清单 + 与其他模块的衔接

**手写能力（不看代码完成）**

- [ ] 写 `ExpertPager`：页表 + 槽位反向索引 + LRU/LFU/FIFO + hit/miss/bytes/stall 记账；
- [ ] 写 `simulate_routing`：每层 Zipf 偏好置换 + Gumbel 噪声（控制 token 间相关性）+ top-k；
- [ ] 复现「淘汰不失效页表 → 双租」bug，并说清为什么命中判定会失真；
- [ ] 复现「9 页循环 + 8 槽位 → LRU 命中率 0」的抖动。

**账本能力（口算 / 手算）**

- [ ] $3\,d\,d_{ff}$ = 176.2 M 参数 → 352 MB @BF16；
- [ ] 32 层 × 8 专家 ≈ 45.1 G 参数 ≈ 90.2 GB；
- [ ] 352 MB ÷ 25 GB/s ≈ 14.1 ms；90 GB ÷ 25 GB/s ≈ 3.6 s；
- [ ] 每 token 全流式 64 个专家 ≈ 902 ms → ≤1.1 token/s。

**表达清晰度（30 秒版）**

- [ ] 专家页 vs KV 页的三个本质差别（生命周期 / 独占性 / 写回）；
- [ ] 为什么「预取 > 淘汰算法」；
- [ ] 为什么 hot expert replication 在稳态下一定掉命中率；
- [ ] 三级存储各层容量、带宽、延迟各差多少数量级。

---

**衔接**：

- 前置：**04-LLM大模型/教学/16-MoE混合专家.ipynb**（路由机制、负载均衡损失、capacity factor）——本节假设你已经知道 top-k 路由怎么来的；
- 同模块：**15-AI-Infra/教学/05-KV Cache与推理内存.ipynb**（KV 页的生命周期，正好与本节的专家页形成对照）、**06-推理引擎vLLM连续批处理与投机解码.ipynb**（连续批处理与投机解码，是「预取 + overlap」的同一套思想）、**02-混合精度与显存账本.ipynb**（BF16 字节数怎么来的）；
- 延伸：**12-智能体开发/教学/02-上下文KV-Cache与Skill.ipynb**（长上下文下的缓存复用）、**13-编程语言/教学/06-Linux系统基础与常用命令.ipynb**（页表 / mmap / page fault 的系统课视角）；
- 一句话记住：**KV 分页省的是「碎片造成的显存浪费」，专家分页省的是「每次 14 ms 的搬运停顿」——同一个页表抽象，两个不同的目标函数。**
